In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from dataclasses import dataclass

pd.set_option("display.max_columns", None)

In [2]:
df = pd .read_csv("../datasets/alzheimers_disease_data.csv")
df.columns = df.columns.str.lower()
df.head()

,patientid,age,gender,ethnicity,educationlevel,bmi,smoking,alcoholconsumption,physicalactivity,dietquality,sleepquality,familyhistoryalzheimers,cardiovasculardisease,diabetes,depression,headinjury,hypertension,systolicbp,diastolicbp,cholesteroltotal,cholesterolldl,cholesterolhdl,cholesteroltriglycerides,mmse,functionalassessment,memorycomplaints,behavioralproblems,adl,confusion,disorientation,personalitychanges,difficultycompletingtasks,forgetfulness,diagnosis,doctorincharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,0,0,1,1,0,0,142,72,242.366840,56.150897,33.682563,162.189143,21.463532,6.518877,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,0,0,0,0,0,0,115,64,231.162595,193.407996,79.028477,294.630909,20.613267,7.118696,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,1,0,0,0,0,0,99,116,284.181858,153.322762,69.772292,83.638324,7.356249,5.895077,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,0,0,0,0,0,0,118,115,159.582240,65.366637,68.457491,277.577358,13.991127,8.965106,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,0,0,0,0,0,0,94,117,237.602184,92.869700,56.874305,291.198780,13.517609,6.045039,0,0,0.014691,0,0,1,1,0,0,XXXConfid


In [3]:
df.drop(columns=["patientid", "doctorincharge"], inplace=True)

In [4]:
df.dtypes.value_counts()

int64      21
float64    12
Name: count, dtype: int64

# **Filter Methods**

#### **1. Variance Threshold** 
In variance threshold we dissect each features solely

In [5]:
def variance(data) -> float:
    total = 0

    # calculate mean
    for num in data:
        total += num
    mean = total / len(data)

    # calculate variance
    var = 0
    for j in data:
        diff_square = (j - mean) ** 2
        var += diff_square

    return var / (len(data) - 1)

In [6]:
variance_scores = {}
for col in df.columns:
    var = variance(df[col])
    variance_scores[col] = var

cv = pd.Series(variance_scores).map(np.sqrt) / df.mean()
cv
# sorted_variances = sorted(variance_scores.items(), key=lambda x: x[1])  
# for feature, var in sorted_variances:
#     print(feature, var) 

age                          0.120016
gender                       0.987744
ethnicity                    1.428071
educationlevel               0.703012
bmi                          0.260975
smoking                      1.570757
alcoholconsumption           0.573529
physicalactivity             0.580706
dietquality                  0.582611
sleepquality                 0.250114
familyhistoryalzheimers      1.722302
cardiovasculardisease        2.436190
diabetes                     2.373887
depression                   1.996981
headinjury                   3.131063
hypertension                 2.391294
systolicbp                   0.193270
diastolicbp                  0.195803
cholesteroltotal             0.188911
cholesterolldl               0.348786
cholesterolhdl               0.389132
cholesteroltriglycerides     0.446759
mmse                         0.583739
functionalassessment         0.569432
memorycomplaints             1.951763
behavioralproblems           2.319344
adl         

In [7]:
cv = cv.sort_values()
cv[cv >= 0.3]

cholesterolldl               0.348786
cholesterolhdl               0.389132
cholesteroltriglycerides     0.446759
functionalassessment         0.569432
alcoholconsumption           0.573529
physicalactivity             0.580706
dietquality                  0.582611
mmse                         0.583739
adl                          0.591973
educationlevel               0.703012
gender                       0.987744
diagnosis                    1.352214
ethnicity                    1.428071
forgetfulness                1.522313
smoking                      1.570757
familyhistoryalzheimers      1.722302
memorycomplaints             1.951763
confusion                    1.968456
depression                   1.996981
difficultycompletingtasks    2.303155
disorientation               2.307177
behavioralproblems           2.319344
diabetes                     2.373887
personalitychanges           2.373887
hypertension                 2.391294
cardiovasculardisease        2.436190
headinjury  

In [8]:
threshold = 0.3
selected_features = []

for feature, var in cv.items():
    if var >= threshold:
        selected_features.append(feature)
print(selected_features)

['cholesterolldl', 'cholesterolhdl', 'cholesteroltriglycerides', 'functionalassessment', 'alcoholconsumption', 'physicalactivity', 'dietquality', 'mmse', 'adl', 'educationlevel', 'gender', 'diagnosis', 'ethnicity', 'forgetfulness', 'smoking', 'familyhistoryalzheimers', 'memorycomplaints', 'confusion', 'depression', 'difficultycompletingtasks', 'disorientation', 'behavioralproblems', 'diabetes', 'personalitychanges', 'hypertension', 'cardiovasculardisease', 'headinjury']


#### **2. Correlation (Pearson, Spearman, Kendall)**
In correlation analysis, we examine features pairwise.

Do these features convey redundant information? If so, we remove one of them to prevent multicollinearity.

In [9]:
# Pearson
def pearson(x, y=df["diagnosis"]):
    import math

    total_x = 0
    for value in x:
        total_x += value
    mean_x = total_x / len(x)

    total_y = 0
    for value in y:
        total_y += value
    mean_y = total_y / len(y)    

    sum_diff_mean = 0
    for x_value, y_value in zip(x, y):
        sum_diff_mean += (x_value - mean_x) * (y_value - mean_y)

    sum_squared_diff_mean = 0
    for x_value in x:
        sum_squared_diff_mean += (x_value - mean_x) ** 2

    sum_squared_diff_mean_y = 0
    for y_value in y:
        sum_squared_diff_mean_y += (y_value - mean_y) ** 2

    denominator = math.sqrt(sum_squared_diff_mean) * math.sqrt(sum_squared_diff_mean_y)

    return sum_diff_mean / denominator

pearson(x=df["bmi"])

0.026342813028879968

In [10]:
my_result = pearson(df["physicalactivity"], df["diagnosis"])
scipy_result = stats.pearsonr(df["physicalactivity"], df["diagnosis"])

print(my_result)
print(scipy_result.statistic)

0.005945042453086641
0.0059450424530866144


In [11]:
# Spearman
def my_rank(data):
    ranks = []

    for value in data:
        smaller = 0
        equal = 0
        for other in data:
            if other < value:
                smaller += 1
            elif other == value: 
                equal += 1
        rank = smaller + (equal + 1) / 2
        ranks.append(rank)
        
    return ranks

def spearman(x, y):
    rank_x = my_rank(x)
    rank_y = my_rank(y)
    return pearson(rank_x, rank_y)

spearman(df["physicalactivity"], df["diagnosis"])

0.0058661315870329415

In [12]:
def my_rank(data):
    sorted_data = sorted(data)
    ranks = {}

    i = 0

    while i < len(sorted_data):
        value = sorted_data[i]
        start = i
        while i < len(sorted_data) and sorted_data[i] == value:
            i += 1
        end = i - 1
        avg_rank = (start + 1 + end + 1) / 2
        ranks[value] = avg_rank
    return [ranks[x] for x in data]

In [13]:
def spearman_formula(x, y):

    rank_x = my_rank(x)
    rank_y = my_rank(y)

    n = len(x)

    sum_d2 = 0

    for rx, ry in zip(rank_x, rank_y):
        d = rx - ry
        sum_d2 += d ** 2

    rho = 1 - (6 * sum_d2) / (n * (n**2 - 1))

    return rho

In [14]:
my_result = spearman(df["physicalactivity"], df["diagnosis"])
scipy_result = stats.spearmanr(df["physicalactivity"], df["diagnosis"])

print(my_result)
print(scipy_result.statistic)

0.0058661315870329415
0.0058661315870329415


####  **3. Chi-Square**
When do we use the Chi-square test?
We generally use it when:
- The **feature** is **categorical**.
- The **target** is also **categorical**.

For instance : (gender, smoking, diabetes, depression, hypertension, familyhistoryalzheimers) are congruous.

**Note on Sample Size:**
The Chi-square test is most reliable with large sample sizes. If the sample size is too small (specifically, if expected frequencies in any cell are less than 5), the results may be inaccurate, and Fisher’s Exact Test should be used instead.

In [15]:
def chi_square(x, y):
    x_levels = sorted(set(x))
    y_levels = sorted(set(y))

    observed = []
    for _ in x_levels:
        observed.append([0] * len(y_levels))

    x_index = {}
    for i, value in enumerate(x_levels):
        x_index[value] = i

    y_index = {}
    for j, value in enumerate(y_levels):
        y_index[value] = j

    for xi, yi in zip(x, y):
        observed[x_index[xi]][y_index[yi]] += 1

    row_totals = []
    for row in observed:
        total = 0
        for value in row:
            total += value
        row_totals.append(total)

    col_totals = []
    for j in range(len(y_levels)):
        total = 0
        for i in range(len(x_levels)):
            total += observed[i][j]
        col_totals.append(total)

    total = 0
    for value in row_totals:
        total += value

    expected = []

    for i in range(len(x_levels)):
        row = []

        for j in range(len(y_levels)):
            e = row_totals[i] * col_totals[j] / total
            row.append(e)

        expected.append(row)

    chi2 = 0

    for i in range(len(x_levels)):
        for j in range(len(y_levels)):
            chi2 += (observed[i][j] - expected[i][j]) ** 2 / expected[i][j]

    dof = (len(x_levels) - 1) * (len(y_levels) - 1)

    return observed, expected, chi2, dof

In [16]:
observed, expected, chi2, dof = chi_square(df["smoking"], df["diagnosis"])

print("Observed:")
for row in observed:
    print(row)

print("\nExpected:")
for row in expected:
    print(row)

print("\nChi2 =", chi2)
print("Degrees of Freedom =", dof)

Observed:
[986, 543]
[403, 217]

Expected:
[988.2647743136342, 540.7352256863658]
[400.73522568636577, 219.26477431363426]

Chi2 = 0.05086793352218713
Degrees of Freedom = 1


In [17]:
from scipy import stats
stats.chi2(dof).sf(chi2)

0.8215598288253867

#### **4. ANOVA**

In [18]:
def create_groups(x, y):
    groups = {}
    for x_value, y_value in zip(x, y):
        if y_value not in groups:
            groups[y_value] = []
        groups[y_value].append(x_value)
    return groups

groups = create_groups(df["age"], df["diagnosis"])

In [19]:
# If the group means differ significantly, ANOVA indicates that this feature is likely important.

def anova(feature, target):

    groups = create_groups(feature, target)
    all_values = []

    for group in groups.values():
        all_values.extend(group)

    grand_mean = np.mean(all_values)

    n = len(all_values)
    k = len(groups)

    # Between
    ss_between = 0
    for group in groups.values():
        group_mean = np.mean(group)
        ss_between += len(group) * (group_mean - grand_mean)**2

    # Within
    ss_within = 0
    for group in groups.values():
        group_mean = np.mean(group)
        for value in group:
            ss_within += (value - group_mean)**2

    # Degrees of freedom
    df_between = k - 1
    df_within = n - k

    # Mean Square
    ms_between = ss_between / df_between
    ms_within = ss_within / df_within

    # F
    F = ms_between / ms_within
    
    return {
        "SS_between": ss_between,
        "SS_within": ss_within,
        "MS_between": ms_between,
        "MS_within": ms_within,
        "F": F, 
        "P_Value" : stats.f(df_between, df_within).sf(F)
    }

In [20]:
result = anova(df["age"], df["diagnosis"])
result = {key: round(float(value), 3) for key, value in result.items()}
result

{'SS_between': 5.23,
 'SS_within': 173604.894,
 'MS_between': 5.23,
 'MS_within': 80.859,
 'F': 0.065,
 'P_Value': 0.799}

#### **5. Mutual Information**
measures how much information a feature provides about the target. Unlike correlation, it captures non-linear dependencies rather than just linear relationships.

In [21]:
import numpy as np
from scipy import stats


# Discretization
def discretize(data, bins=10):

    minimum = min(data)
    maximum = max(data)

    width = (maximum - minimum) / bins

    result = []

    for value in data:
        if value == maximum:
            result.append(bins-1)
        else:
            result.append(int((value-minimum)/width))

    return result


def is_numeric(data):
    for value in data:
        if not isinstance(value,(int,float,np.number)):
            return False
    return True



# Entropy
def entropy(data, base=2):
    values, counts = np.unique(data, return_counts=True)
    probabilities = counts / len(data)
    h = 0
    for p in probabilities:
        h -= p * (np.log(p) / np.log(base))
    return h, len(values)


# Joint Entropy
def joint_entropy(x,y,base=2):

    table = {}

    for x_value,y_value in zip(x,y):
        if x_value not in table:
            table[x_value] = {}
        if y_value not in table[x_value]:
            table[x_value][y_value] = 0
        table[x_value][y_value]+=1

    total = len(x)
    h = 0
    for x_value in table:
        for y_value in table[x_value]:
            p = table[x_value][y_value] / total
            h -= p * (np.log(p) / np.log(base))
    return h


# Mutual Information
@dataclass
class MutualInformation:
    mutual_information: float
    entropy_joint: float
    entropy_x: float
    entropy_y: float
    levels_x: int
    levels_y: int
    observations: int

    @classmethod
    def fit(cls, X,Y):
        hx,nx = entropy(X)
        hy,ny = entropy(Y)
        hxy = joint_entropy(X,Y)
        mi = hx + hy - hxy

        return cls(mi, hxy, hx, hy, nx, ny, len(X))

    def normalized_mi(self):
        denominator = np.sqrt(self.entropy_x * self.entropy_y)
        return self.mutual_information / denominator

    def adjusted_mi(self):
        expected = ((self.levels_x-1) * (self.levels_y-1) / (2*self.observations*np.log(2)))
        maximum = min(np.log2(self.levels_x), np.log2(self.levels_y))
        ami = (self.mutual_information - expected) / (maximum - expected)
        return max(0,min(1,ami))

    def statistical_test(self):
        g2 = ( 2 * self.observations * np.log(2) * self.mutual_information)


    def summary(self):
        return {
            "MI": round(self.mutual_information, 3),
            "Normalized MI": round(self.normalized_mi(), 3),
            "Adjusted MI": round(self.adjusted_mi(), 3),
            "Entropy X": round(self.entropy_x, 3),
            "Entropy Y": round(self.entropy_y, 3),
            "Joint Entropy": round(self.entropy_joint, 3),
            "Levels X": self.levels_x,
            "Levels Y": self.levels_y,
            "Observations": self.observations,
        }

# Final General Function
def mutual_information(feature, target, bins=10):
    if is_numeric(feature):
        feature = discretize(feature, bins)

    result = MutualInformation.fit(feature, target)

    return result

In [22]:
mi_age = mutual_information(df["age"], df["diagnosis"])
mi_age.summary()

{'MI': 0.001,
 'Normalized MI': 0.001,
 'Adjusted MI': 0,
 'Entropy X': 3.306,
 'Entropy Y': 0.937,
 'Joint Entropy': 4.242,
 'Levels X': 10,
 'Levels Y': 2,
 'Observations': 2149}

#  TODO: Feature Selection
##  Filter Methods (Completed)

- [x] Variance Threshold
- [x] Pearson Correlation
- [x] Spearman Correlation
- [x] Chi-Square Test
- [x] ANOVA (F-Test)
- [x] Mutual Information

---

##  Wrapper Methods (To Do)

- [ ] Sequential Forward Selection (SFS)
- [ ] Sequential Backward Selection (SBS)
- [ ] Recursive Feature Elimination (RFE)

---

## Embedded Methods (To Do)

- [ ] Decision Tree Feature Importance
- [ ] Random Forest Feature Importance
- [ ] Lasso (L1 Regularization)
- [ ] Ridge (L2 Regularization)


---

## Goal

Implement every algorithm **from scratch first**, then compare the results with **scikit-learn** and other libraries.